# Garmin Data Exploration & Cleaning

Inspect the raw data pulled from Garmin Connect, identify quality issues, and decide what to clean or keep.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src.db import load_dataframe

# Load RAW data (no filters) to see everything
raw = load_dataframe(raw=True)
clean = load_dataframe(raw=False)
print(f"Raw: {len(raw)} activities")
print(f"Clean (after filters): {len(clean)} activities")
print(f"Filtered out: {len(raw) - len(clean)} activities")

## 1. Schema & Column Overview
What columns do we have, what types, how many nulls?

In [ ]:
print("=== Column Types ===")
print(raw.dtypes)
print(f"\n=== Shape: {raw.shape[0]} rows x {raw.shape[1]} columns ===")
print(f"\nDate range: {raw['start_time'].min()} to {raw['start_time'].max()}")

In [ ]:
# Null counts per column
nulls = raw.isnull().sum()
null_pct = (nulls / len(raw) * 100).round(1)
pd.DataFrame({"nulls": nulls, "pct": null_pct}).query("nulls > 0").sort_values("nulls", ascending=False)

## 2. Basic Statistics
Descriptive stats for all numeric columns — look for suspicious min/max values.

In [ ]:
raw[["distance_km", "duration_min", "pace_min_km", "calories",
     "avg_hr", "max_hr", "avg_speed", "elevation_gain", "cadence"]].describe().round(2)

## 3. Outlier Detection
Spot activities with extreme values that may be GPS glitches, indoor treadmill artifacts, or partial recordings.

In [ ]:
# Shortest activities (potential fragments)
print("=== Shortest by distance (< 1 km) ===")
short = raw[raw["distance_km"] < 1].sort_values("distance_km")
print(f"{len(short)} activities")
short[["name", "start_time", "distance_km", "duration_min", "pace_min_km"]].head(10)

In [ ]:
# Extreme paces (too fast or too slow)
print("=== Pace < 2 min/km (impossibly fast) ===")
too_fast = raw[raw["pace_min_km"] < 2]
print(f"{len(too_fast)} activities")
if len(too_fast): display(too_fast[["name", "start_time", "distance_km", "duration_min", "pace_min_km"]].head())

print("\n=== Pace > 15 min/km (walking pace or GPS issues) ===")
too_slow = raw[raw["pace_min_km"] > 15]
print(f"{len(too_slow)} activities")
if len(too_slow): display(too_slow[["name", "start_time", "distance_km", "duration_min", "pace_min_km"]].head())

In [ ]:
# Heart rate anomalies
print("=== Missing heart rate ===")
no_hr = raw[raw["avg_hr"].isna()]
print(f"{len(no_hr)} activities with no HR data ({len(no_hr)/len(raw)*100:.1f}%)")

print("\n=== Suspicious HR (avg < 80 or max > 220) ===")
weird_hr = raw[(raw["avg_hr"] < 80) | (raw["max_hr"] > 220)].dropna(subset=["avg_hr"])
print(f"{len(weird_hr)} activities")
if len(weird_hr): display(weird_hr[["name", "start_time", "avg_hr", "max_hr", "distance_km", "pace_min_km"]].head())

In [ ]:
# Cadence anomalies
print("=== Missing cadence ===")
no_cad = raw[raw["cadence"].isna()]
print(f"{len(no_cad)} activities with no cadence data ({len(no_cad)/len(raw)*100:.1f}%)")

print("\n=== Unusual cadence (< 130 or > 210 spm) ===")
weird_cad = raw[(raw["cadence"] < 130) | (raw["cadence"] > 210)].dropna(subset=["cadence"])
print(f"{len(weird_cad)} activities")
if len(weird_cad): display(weird_cad[["name", "start_time", "cadence", "distance_km", "pace_min_km"]].head())

## 4. Distributions
Histograms to visually spot where the data clusters and where outliers live.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle("Raw Data Distributions", fontsize=14, fontweight="bold")

plots = [
    ("distance_km", "Distance (km)", "#2196F3"),
    ("duration_min", "Duration (min)", "#4CAF50"),
    ("pace_min_km", "Pace (min/km)", "#FF5722"),
    ("calories", "Calories", "#FF9800"),
    ("avg_hr", "Avg HR (bpm)", "#E91E63"),
    ("max_hr", "Max HR (bpm)", "#9C27B0"),
    ("elevation_gain", "Elevation (m)", "#795548"),
    ("cadence", "Cadence (spm)", "#607D8B"),
    ("avg_speed", "Avg Speed (m/s)", "#00BCD4"),
]

for ax, (col, label, color) in zip(axes.flat, plots):
    data = raw[col].dropna()
    ax.hist(data, bins=40, color=color, alpha=0.7, edgecolor="white")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.axvline(data.median(), color="black", linestyle="--", alpha=0.5, label=f"median: {data.median():.1f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. The Filtered Records
Inspect exactly what the current cleaning filters remove. Are we losing anything valuable?

In [ ]:
# What gets filtered out?
filtered_out = raw[~raw["activity_id"].isin(clean["activity_id"])]
print(f"{len(filtered_out)} records removed by current filters:\n")
print("Current filter rules:")
print("  - distance_km >= 0.5")
print("  - duration_s > 0")
print("  - pace between 2-15 min/km")

print(f"\nBreakdown of why they were filtered:")
print(f"  Zero/near-zero distance (< 0.5 km): {len(raw[raw['distance_km'] < 0.5])}")
print(f"  Zero duration: {len(raw[raw['duration_s'] <= 0])}")
print(f"  Pace too fast (< 2 min/km): {len(raw[raw['pace_min_km'] < 2])}")
print(f"  Pace too slow (> 15 min/km): {len(raw[raw['pace_min_km'] > 15])}")

In [ ]:
# Show all filtered records for inspection
filtered_out[["name", "start_time", "distance_km", "duration_min",
              "pace_min_km", "avg_hr", "calories"]].sort_values("start_time")

## 6. Activity Names
Check for naming patterns, duplicates, or oddities.

In [ ]:
print(f"Unique activity names: {raw['name'].nunique()}")
print(f"\nMost common names:")
raw["name"].value_counts().head(15)

## 7. Time Gaps & Frequency
How consistently have you been running? Any long gaps?

In [ ]:
clean_sorted = clean.sort_values("start_time").reset_index(drop=True)
gaps = clean_sorted["start_time"].diff().dt.days

print(f"Average days between runs: {gaps.mean():.1f}")
print(f"Median days between runs: {gaps.median():.1f}")
print(f"Longest gap: {gaps.max():.0f} days")

print(f"\n=== Gaps longer than 14 days ===")
long_gaps = gaps[gaps > 14]
for idx in long_gaps.index:
    row = clean_sorted.iloc[idx]
    prev = clean_sorted.iloc[idx - 1]
    print(f"  {int(gaps[idx])}d gap — last run: {prev['start_time'].strftime('%Y-%m-%d')} → next: {row['start_time'].strftime('%Y-%m-%d')}")

In [ ]:
# Runs per month over time
monthly = clean.set_index("start_time").resample("ME").size()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(monthly.index, monthly.values, width=25, color="#2196F3", alpha=0.7)
ax.set_ylabel("Runs")
ax.set_title("Runs Per Month")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Year-over-Year Comparison
Quick view of volume by year.

In [ ]:
clean["year"] = clean["start_time"].dt.year
yearly = clean.groupby("year").agg(
    runs=("activity_id", "count"),
    total_km=("distance_km", "sum"),
    avg_pace=("pace_min_km", "mean"),
    avg_hr=("avg_hr", "mean"),
    total_elevation=("elevation_gain", "sum"),
    total_calories=("calories", "sum"),
).round(1)
yearly

## Next Steps

After reviewing the data above, decide:
1. **Adjust filter thresholds?** (currently: distance >= 0.5km, pace 2-15 min/km)
2. **Handle nulls?** (HR and cadence may have gaps from older watches)
3. **Remove specific activities?** (mark individual IDs for exclusion)
4. **Add new derived columns?** (e.g. pace zones, time of day, season)